# ML5 Supervised Learning. Decision Trees and ensembles

### IMPORT

In [1]:
import pandas as pd
import numpy as np
import sys
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, precision_score,recall_score,f1_score, roc_curve, auc, roc_auc_score, average_precision_score,accuracy_score
import optuna


### 1. Download data from Don’tGetKicked competition. Design train/validation/test split.

Use the "PurchDate" field to split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.

Use the first 33% of the data for the training, the last 33% of the data for the test, and the middle 33% for the validation set. Don't use the test dataset until the end!

Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training), for example: https://contrib.scikit-learn.org/category_encoders/count.html

In [2]:
df = pd.read_csv('data/training.csv',parse_dates=['PurchDate'])

In [3]:
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


#### 1.1 Design the train/validation/test split. 

* Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set. Don’t use the test dataset until the end!

In [4]:
def my_date_val_split(df, validation_date, test_date, date_col='created'):

    if len(df) == 0:
        print('empty df')
        sys.exit(1)

    if len(df) < 2:
        print('len df should be more than 1')
        sys.exit(1)

    try:
        dt_val_split = pd.to_datetime(validation_date, errors='raise')
    except:
        print(f'validation_date: {validation_date} NOT CONVERTED GOOD TO DT FORMAT')
        sys.exit(1)

    try:
        dt_test_split = pd.to_datetime(test_date, errors='raise')
    except:
        print(f'test_date: {test_date} NOT CONVERTED GOOD TO DT FORMAT')
        sys.exit(1)


    max_date = df[date_col].max()
    min_date = df[date_col].min()

    if dt_val_split > max_date or dt_val_split < min_date:
        print('Date for split is after to max datetime OR vice versa less than min datetime')
        sys.exit(1)
    
    if dt_test_split > max_date or dt_test_split < min_date:
        print('Date for split is after to max datetime OR vice versa less than min datetime')
        sys.exit(1)

    # if dt_val_split > dt_test_split:
    #     print('Error val_date must be less than test_date')
    #     sys.exit(1)

    df_sorted = df.sort_values(by=date_col,ascending=True)
    
    train_split = df_sorted[df_sorted[date_col] < dt_val_split]

    val_split = df_sorted[
        (df_sorted[date_col] >= dt_val_split) & (df_sorted[date_col] < dt_test_split)
        ]
    test_split = df_sorted[df_sorted[date_col] >= dt_test_split]

    return (train_split,val_split,test_split)



In [5]:
# val_date = df.sort_values(by='PurchDate',ascending=True).loc[int(len(df)*0.34),'PurchDate']
# test_date = df.sort_values(by='PurchDate',ascending=True).loc[int(len(df)*0.67),'PurchDate']
val_date = '2009-10-01'  
test_date = '2010-05-01' 
print(val_date,test_date)

2009-10-01 2010-05-01


In [6]:
df.set_index('RefId',inplace=True)

In [7]:
df_train,df_val,df_test = my_date_val_split(df,val_date,test_date,'PurchDate')

In [8]:
df_train.shape,df_val.shape,df_test.shape

((26303, 33), (20860, 33), (25820, 33))

#### 1.2 Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables

* Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html

In [9]:
class MyTypeChanger(BaseEstimator,TransformerMixin):
    
    def __init__(self):
        self.dtype_dict = {}
        with open('data/data_dtype.csv', 'r') as f:
            for line in f:
                key, value = line.strip().split(',')
                self.dtype_dict[key] = value

    def fit(self,df,y=None):
        
        #Удаление ключей которых нет в датафрейме
        # for key in self.dtype_dict.keys():
        #     if key not in df.columns:
        #         del self.dtype_dict[key]
        
        return self
    
    def transform(self,df):
        output_df = df.copy()
        for col in df.columns:
            if col in self.dtype_dict.keys():

                if self.dtype_dict[col] == 'drop':
                    output_df = output_df.drop(columns = col)

                elif self.dtype_dict[col] == 'int' or self.dtype_dict[col] == 'float':
                    # output_df[col] = output_df[col].astype('int64')
                    output_df[col] = output_df[col].apply(pd.to_numeric, errors='coerce')

                # elif self.dtype_dict[col] == 'float':
                #     output_df[col] = output_df[col].astype('float64')
                #     output_df[col] = output_df[col].apply(pd.to_numeric, errors='coerce')

                elif self.dtype_dict[col] == 'category':
                    output_df[col] = output_df[col].astype('category')
                
            else:
                print(f'Col {col} not transformed')

        return output_df

In [10]:
class MySimpleImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='median', cat_strategy='special_value'):
        self.strategy = strategy
        self.cat_strategy = cat_strategy
        self.fill_values_ = {} # Здесь будем хранить значения для каждого столбца

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.fill_values_ = {}
        
        for col in X.columns:
            # Логика для категориальных признаков
            if X[col].dtype == 'object' or isinstance(X[col].dtype, pd.CategoricalDtype):
                if self.cat_strategy == 'most_common':
                    self.fill_values_[col] = X[col].mode()[0]
                elif self.cat_strategy == 'special_value':
                    self.fill_values_[col] = 'MISSINGVALUE'
                # 'drop' не требует сохранения значения
            
            # Логика для числовых признаков
            else:
                if self.strategy == 'median':
                    self.fill_values_[col] = X[col].median()
                elif self.strategy == 'mean':
                    self.fill_values_[col] = X[col].mean()
                elif self.strategy == 'min':
                    self.fill_values_[col] = X[col].min()
                elif self.strategy == 'max':
                    self.fill_values_[col] = X[col].max()
                    
        return self

    def transform(self, X):
        X = X.copy()
        
        # Сначала обрабатываем удаление, если выбрано
        if self.cat_strategy == 'drop':
            cat_cols = X.select_dtypes(include=['object', 'category']).columns
            X = X.dropna(subset=cat_cols)

        # Заполняем пропуски сохраненными значениями
        for col, value in self.fill_values_.items():
            if col in X.columns:
                X[col] = X[col].fillna(value)
                
        return X


In [11]:
numeric_features = []
categorical_features = []
with open('data/data_dtype.csv', 'r') as f:
    for line in f:
        key, value = line.strip().split(',')
        if value in ['int','float']:
            numeric_features.append(key)
        elif value in ['object','category','string']:
            categorical_features.append(key)

numeric_transformer = StandardScaler()

categorical_transformer = OrdinalEncoder(handle_unknown = 'use_encoded_value',unknown_value = -1)

col_trans = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

In [12]:
preprocessing = Pipeline([
    ('my_type_changer', MyTypeChanger()),
    ('my_simple_imputer', MySimpleImputer()),
    ('col_trans',col_trans)
      ])

In [13]:
X_train, y_train = df_train.drop(columns='IsBadBuy'),df_train['IsBadBuy']
X_valid, y_valid = df_val.drop(columns='IsBadBuy'),df_val['IsBadBuy']
X_test, y_test = df_test.drop(columns='IsBadBuy'),df_test['IsBadBuy']

In [14]:
X_train_trans = preprocessing.fit(X=X_train).transform(X=X_train)
feature_names = preprocessing.named_steps['col_trans'].get_feature_names_out()
X_train_trans = pd.DataFrame(X_train_trans, columns=feature_names,index=df_train.index)

for col in X_train_trans.columns:
    if 'cat' in col:
        X_train_trans[col] = pd.to_numeric(X_train_trans[col],errors='raise',downcast='integer')

In [15]:
X_valid_trans = preprocessing.fit(X=X_train).transform(X=X_valid)
X_valid_trans = pd.DataFrame(X_valid_trans, columns=feature_names,index=df_val.index)

for col in X_valid_trans.columns:
    if 'cat' in col:
        X_valid_trans[col] = pd.to_numeric(X_valid_trans[col],errors='raise',downcast='integer')

In [16]:
X_test_trans = preprocessing.fit(X=X_train).transform(X=X_test)
X_test_trans = pd.DataFrame(X_test_trans, columns=feature_names,index=df_test.index)

for col in X_test_trans.columns:
    if 'cat' in col:
        X_test_trans[col] = pd.to_numeric(X_test_trans[col],errors='raise',downcast='integer')

### 2. Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).

The Gini Coefficient is a metric used to measure inequality or impurity in datasets. In machine learning, especially in decision trees, it quantifies how mixed the classes are in a node. A Gini index of 0 means perfect purity (all elements belong to one class) and a higher Gini index indicates more impurity.

$$ Gini(p) = 1 - \sum_{i=0}^N{p_i^2} $$



In information theory, the entropy of a random variable quantifies the average level of uncertainty or information associated with the variable's potential states or possible outcomes. This measures the expected amount of information needed to describe the state of the variable, considering the distribution of probabilities across all potential states.

$$ H(p) = - \sum_{i=0}^N{p_i \log_2{p_i}} $$



Laplas smoothing:

$$ P(c) = \frac{count(c) + 1}  {len(data\_idx) + len(c)}  $$

It should support fit, predict_proba, and predict methods. Also, the maximum depth (max_depth) must be a parameter of your class. Use the Gini impurity criterion as a criterion for choosing the split.

Here is the blueprint:  

model = DecisionTreeClassifier(max_depth=7)  
model.fit(Xtrain, ytrain)  
model.predict_proba(Xvalid)  

* Create a separate class for Node. It should be able to hold data (sample features and targets), compute Gini impurity, and have pointers to children (left and right nodes). For the Regressor, use standard deviation instead of Gini impurity.
* Implement a function that finds the best possible split in the current node.
* Combine the previous steps into your working Decision Tree Classifier.
* Implement an Extra Randomized Tree by designing another function to find the best split.

In [17]:
categorical_features = []
for col in X_train_trans.columns:
    if 'cat' in col:
        categorical_features.append(col)

In [18]:
def entropy(p):
    eps = 1e-15
    return -np.sum(p * np.log2(p + eps))

In [19]:
def gini_criteria(y):
    y = pd.Series(y)
    prob = y.value_counts(normalize=True)
    return 1 - np.sum(np.square(prob))

In [20]:
class Node:
    def __init__(self,feature_idx = None,threshold = None, left = None,right = None, is_leaf = None,value = None,is_cat = None, gain = None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.is_leaf = is_leaf
        self.value = value
        self.is_cat = is_cat
        self.gain = gain


In [21]:
class MyDecisionTree:
    def __init__(self, max_depth, cat_feat, min_samples_split = 2, criteria = 'gini',split = 'best', n_treshholds = 30, sample_weights = None):

        # Проверка на валидное значение глубины
        if max_depth >= 1: 
            self.max_depth = max_depth
        else:
            print(f"Max depth must be more than 0 got {max_depth}")
            sys.exit(1)

        if min_samples_split < 2:
            print(f"min_samples_split must be more than 2 got {min_samples_split}")
            sys.exit(1)
        else:
            self.min_samples_split = min_samples_split
            
        if criteria not in ['entropy','gini']:
            print('Error criteria must be in [\'entropy\',\'gini\']')
        else:
            self.criteria = criteria
        
        if split not in ['best','random']:
            print('Error split must be in [\'best\',\'random\']')
        else:
            self.split = split

        self.cat_feat = cat_feat
        
        self.n_treshholds = n_treshholds

        if sample_weights not in ['balanced'] and not isinstance(sample_weights, dict) and sample_weights is not None:
            print('sample_weights must be \'balanced\' or dict')
            sys.exit(1)
        else:
            self.sample_weights = sample_weights

    def fit(self,X,y):
        self.X = X
        self.y = y
        self.classes_ = np.sort(np.unique(y))

        if self.sample_weights in ['balanced']:
            total_elems = len(y)
            n_classes = len(self.classes_)
            count = self.y.value_counts()
            balanced_dict = total_elems / (n_classes * count)
            self.weights_vector = self.y.map(balanced_dict)

        elif isinstance(self.sample_weights, dict):
            self.weights_vector = self.y.map(self.sample_weights)

        else:
            self.weights_vector = pd.Series(1, index = self.y.index)
            
        self.weights_vector.fillna(1,inplace = True)

        self._make_numerical_thresholds()
        self.root = self._build_tree(depth=1, data_idx = self.X.index)

        
    # def _laplas_smoothing(self,data_idx):
    #     current_weights = self.weights_vector.loc[data_idx]
    #     full_weights_count = pd.Series(0,index = self.classes_)
    #     weighted_counts = self.weights_vector.groupby(self.y.loc[data_idx]).sum()
    #     total_weight = current_weights.sum()
    #     full_weights_count[weighted_counts.index] =weighted_counts.values
    #     return (full_weights_count + 1) / (total_weight + len(self.classes_))

    def _laplas_smoothing(self, data_idx):
        current_weights = self.weights_vector.loc[data_idx]
        
        # Считаем веса для каждого класса и сразу переиндексируем под self.classes_
        weighted_counts = (
            self.weights_vector.groupby(self.y.loc[data_idx])
            .sum()
            .reindex(self.classes_, fill_value=0)
        )
        
        total_weight = current_weights.sum()
        # Возвращаем Series сглаженных вероятностей
        return (weighted_counts + 1) / (total_weight + len(self.classes_))
    
    def _build_tree(self,depth, data_idx):
        
        prob_vector = self._laplas_smoothing(data_idx)

        if depth > self.max_depth:
            return Node(is_leaf=True, value = prob_vector)
        
        if self.min_samples_split > len(data_idx):
            return Node(is_leaf=True, value = prob_vector)
        
        if entropy(prob_vector) < 1e-7:
            return Node(is_leaf=True, value=prob_vector)
        
        if self.split == 'best':
            best_feat,best_thresh,inform_gain = self._find_best_split(data_idx)
            
        elif self.split == 'random':
            best_feat,best_thresh,inform_gain = self._find_random_split(data_idx)

        if inform_gain <= 0 or best_feat is None or best_thresh is None:
            return Node(is_leaf = True, value = prob_vector)
        

        if best_feat in self.cat_feat:
            feat_values = self.X.loc[data_idx, best_feat]
            left_indx = data_idx[feat_values.to_numpy() == best_thresh]
            right_indx = data_idx[feat_values.to_numpy() != best_thresh]
            is_cat = True
        else:
            feat_values = self.X.loc[data_idx, best_feat]
            left_indx = data_idx[feat_values.to_numpy() <= best_thresh]
            right_indx = data_idx[feat_values.to_numpy() > best_thresh]
            is_cat = False


        left_node = self._build_tree(depth=depth+1, data_idx=left_indx)
        right_node = self._build_tree(depth=depth+1, data_idx=right_indx)

        return Node(feature_idx = best_feat, threshold = best_thresh, left = left_node, right = right_node, is_leaf = False, value = None, is_cat=is_cat, gain = inform_gain)
    
    def _find_best_split(self,data_idx):

        best_feat, best_thresh, inform_gain = None, None, -1

        for feat in self.X.columns:

            if feat in self.cat_feat:
                feat_is_categorical = True
            else:
                feat_is_categorical = False

            if feat_is_categorical: # проверяешь по списку категориальных колонок
                # Перебираем уникальные значения для == и !=
                feat_values = self.X.loc[data_idx, feat]

                for value in feat_values.unique():
                    # Считаем gain для сплита (X[feat] == value)
                    
                    left = data_idx[feat_values.to_numpy() == value]
                    right = data_idx[feat_values.to_numpy() != value]

                    iter_gain = self._calc_gain(data_idx,right=right,left=left)

                    if iter_gain > inform_gain:
                        best_feat, best_thresh, inform_gain = feat, value, iter_gain

            else:
                # Числовой признак: генерируем фиксированную сетку порогов
                thresholds = self._get_numerical_thresholds(feat)
                feat_values = self.X.loc[data_idx, feat]
                
                for thresh in thresholds:
                    # Считаем gain для сплита (X[feat] <= thresh)
                    
                    left = data_idx[feat_values.to_numpy() <= thresh]
                    right = data_idx[feat_values.to_numpy() > thresh]

                    iter_gain = self._calc_gain(data_idx,right=right,left=left)
                    if iter_gain > inform_gain:
                        best_feat, best_thresh, inform_gain = feat, thresh, iter_gain
            
        return best_feat, best_thresh, inform_gain
    

    def _find_random_split(self,data_idx):

        feat = np.random.choice(self.X.columns)
        feat_values = self.X.loc[data_idx, feat]

        if feat in self.cat_feat:
            thresh = np.random.choice(feat_values.unique())
            left = data_idx[feat_values.to_numpy() == thresh]
            right = data_idx[feat_values.to_numpy() != thresh]

        else:
            thresh = np.random.uniform(feat_values.min(), feat_values.max()) # Выбор рандомного локального значения из min - max выборки
            left = data_idx[feat_values.to_numpy() <= thresh]
            right = data_idx[feat_values.to_numpy() > thresh]

        inform_gain = self._calc_gain(data_idx,right=right,left=left)
        return feat, thresh, inform_gain


    def _make_numerical_thresholds(self):
        self.thresh_holds = {}

        for feat in self.X.select_dtypes(include=['number']):
            values = []

            for q in np.linspace(1,99,num=self.n_treshholds):
                values.append(np.percentile(self.X[feat],q=q))

            self.thresh_holds[feat] = values
    
    def _get_numerical_thresholds(self,feat):
        return self.thresh_holds[feat]

    def _calc_gain(self, data_idx, right, left):

        current_weights = self.weights_vector.loc[data_idx]

        if len(right) == 0 or len(left) ==0:
            return 0

        if self.criteria == 'entropy': ################# :c 

            # расчёт взвешенной вероятности!!! 
            parent_prob = self._calc_weight_prob(data_idx)
            left_prob = self._calc_weight_prob(left)
            right_prob = self._calc_weight_prob(right)

            # считаем энтропию по взвешенной вероятности!!!
            parent_entr = entropy(parent_prob)
            h_right = entropy(right_prob) * len(right) / len(data_idx)
            h_left = entropy(left_prob) * len(left) / len(data_idx)
            gain = parent_entr - (h_left + h_right)

        if self.criteria == 'gini':
            parent_gini = gini_criteria(self.y[data_idx])
            h_right = gini_criteria(self.y[right]) * len(right) / len(data_idx)
            h_left = gini_criteria(self.y[left]) * len(left) / len(data_idx)
            gain = parent_gini - (h_left + h_right)

        return gain
    
    def _calc_weight_prob(self,data_idx):
        total_weights = self.weights_vector.loc[data_idx].sum()
        class_weights = self.weights_vector.groupby(self.y[data_idx]).sum()
        prob = class_weights / total_weights
        return prob
    
    def predict(self,X_test):
        prob = self.predict_proba(X_test)
        return  prob.idxmax(axis=1)
    
    def predict_proba(self,X_test):
        y_pred = pd.DataFrame(index=X_test.index, columns=self.classes_,dtype=np.float64)
        self.X_test = X_test
        self._pred_node(self.root,X_test.index,y_pred)
        return y_pred

    def _pred_node(self, node, current_ind, preds):
        if len(current_ind) == 0:
            return
        if node.is_leaf:
            preds.loc[current_ind,:] = node.value.values
            return
        else:
            
            if node.is_cat:
                feat_values = self.X_test.loc[current_ind, node.feature_idx]
                left_indx = current_ind[feat_values.to_numpy() == node.threshold]
                right_indx = current_ind[feat_values.to_numpy() != node.threshold]
                self._pred_node(node.left,left_indx, preds)
                self._pred_node(node.right,right_indx, preds)

            else:
                feat_values = self.X_test.loc[current_ind, node.feature_idx]
                left_indx = current_ind[feat_values.to_numpy() <= node.threshold]
                right_indx = current_ind[feat_values.to_numpy() > node.threshold]
                self._pred_node(node.left,left_indx, preds)
                self._pred_node(node.right,right_indx, preds)


        

In [22]:
class MyDecisionTreeRegressor:
    def __init__(self, max_depth, cat_features, min_samples_split = 2, criteria = 'variance',split = 'best', n_treshholds = 30):

        # Проверка на валидное значение глубины
        if max_depth >= 1: 
            self.max_depth = max_depth
        else:
            print(f"Max depth must be more than 0 got {max_depth}")
            sys.exit(1)

        if min_samples_split < 2:
            print(f"min_samples_split must be more than 2 got {min_samples_split}")
            sys.exit(1)
        else:
            self.min_samples_split = min_samples_split
            
        if criteria not in ['variance']:
            print('Error criteria must be in [\'variance\']')
        else:
            self.criteria = criteria
        
        if split not in ['best']:
            print('Error split must be in [\'best\']')
        else:
            self.split = split

        self.cat_features = cat_features
        self.n_treshholds = n_treshholds
    
    def fit(self,X,y):
        self.X = X
        self.y = y
        self.classes_ = np.sort(np.unique(y))
        self._make_numerical_thresholds()
        self.root = self._build_tree(depth=1, data_idx = self.X.index)
        
        
    def _build_tree(self, depth, data_idx):
        
        output = self.y[data_idx].mean()

        if depth > self.max_depth:
            return Node(is_leaf=True, value = output)
        
        if self.min_samples_split > len(data_idx):
            return Node(is_leaf=True, value = output)
        
        if np.var(self.y.loc[data_idx]) < 1e-7:
            return Node(is_leaf=True, value=output)
        
        if self.split == 'best':
            best_feat,best_thresh,inform_gain = self._find_best_split(data_idx)

        if inform_gain <= 0 or best_feat is None or best_thresh is None:
            return Node(is_leaf = True, value = output)
        
        if best_feat in self.cat_features:
            feat_values = self.X.loc[data_idx, best_feat]
            left_indx = data_idx[feat_values.to_numpy() == best_thresh]
            right_indx = data_idx[feat_values.to_numpy() != best_thresh]
            is_cat = True
        else:
            feat_values = self.X.loc[data_idx, best_feat]
            left_indx = data_idx[feat_values.to_numpy() <= best_thresh]
            right_indx = data_idx[feat_values.to_numpy() > best_thresh]
            is_cat = False


        left_node = self._build_tree(depth=depth+1, data_idx=left_indx)
        right_node = self._build_tree(depth=depth+1, data_idx=right_indx)

        return Node(feature_idx = best_feat, threshold = best_thresh, left = left_node, right = right_node, is_leaf = False, value = None, is_cat=is_cat, gain = inform_gain)
    
    def _find_best_split(self,data_idx):

        best_feat, best_thresh, inform_gain = None, None, -1

        for feat in self.X.columns:

            if feat in self.cat_features: # проверяешь по списку категориальных колонок
                # Перебираем уникальные значения для == и !=
                feat_values = self.X.loc[data_idx, feat]

                for value in feat_values.unique():
                    # Считаем gain для сплита (X[feat] == value)
                    
                    left = data_idx[feat_values.to_numpy() == value]
                    right = data_idx[feat_values.to_numpy() != value]

                    iter_gain = self._calc_gain(data_idx,right=right,left=left)

                    if iter_gain > inform_gain:
                        best_feat, best_thresh, inform_gain = feat, value, iter_gain

            else:
                # Числовой признак: генерируем фиксированную сетку порогов
                thresholds = self._get_numerical_thresholds(feat)
                feat_values = self.X.loc[data_idx, feat]
                
                for thresh in thresholds:
                    # Считаем gain для сплита (X[feat] <= thresh)
                    
                    left = data_idx[feat_values.to_numpy() <= thresh]
                    right = data_idx[feat_values.to_numpy() > thresh]

                    iter_gain = self._calc_gain(data_idx,right=right,left=left)
                    if iter_gain > inform_gain:
                        best_feat, best_thresh, inform_gain = feat, thresh, iter_gain
            
        return best_feat, best_thresh, inform_gain

    def _make_numerical_thresholds(self):
        self.thresh_holds = {}  

        for feat in self.X.select_dtypes(include=['number']):
            values = []

            for q in np.linspace(1,99,self.n_treshholds):
                values.append(np.percentile(self.X[feat],q=q))

            self.thresh_holds[feat] = values

    def _get_numerical_thresholds(self,feat):
        return self.thresh_holds[feat]  

    def _calc_gain(self, data_idx, right, left):

        if len(right) < 2 or len(left) < 2:
            return 0

        if self.criteria == 'variance':            # ЗДЕСЬ ВСЕ ЧЕТКО
            parent_var = np.var(self.y[data_idx])
            var_right = np.var(self.y[right]) * len(right) / len(data_idx)
            var_left = np.var(self.y[left]) * len(left) / len(data_idx)
            gain = parent_var - (var_left + var_right)

        return gain
    
    def predict(self,X_test):
        y_pred = pd.Series(index=X_test.index,dtype=np.float64)
        self.X_test = X_test
        self._pred_node(self.root,X_test.index,y_pred)
        return  y_pred
    

    def _pred_node(self, node, current_ind, preds):
        if len(current_ind) == 0:
            return
        if node.is_leaf:
            preds.loc[current_ind] = node.value
            return
        else:
            
            if node.is_cat:
                feat_values = self.X_test.loc[current_ind, node.feature_idx]
                left_indx = current_ind[feat_values.to_numpy() == node.threshold]
                right_indx = current_ind[feat_values.to_numpy() != node.threshold]
                self._pred_node(node.left,left_indx, preds)
                self._pred_node(node.right,right_indx, preds)

            else:
                feat_values = self.X_test.loc[current_ind, node.feature_idx]
                left_indx = current_ind[feat_values.to_numpy() <= node.threshold]
                right_indx = current_ind[feat_values.to_numpy() > node.threshold]
                self._pred_node(node.left,left_indx, preds)
                self._pred_node(node.right,right_indx, preds)


In [23]:
model = MyDecisionTree(max_depth = 5,cat_feat=categorical_features,criteria='entropy',sample_weights='balanced')
model.fit(X_train_trans,y_train)
y_pred_prob = model.predict_proba(X_valid_trans)

In [24]:
y_pred_prob[1].values

array([0.27027312, 0.46990818, 0.46990818, ..., 0.53782766, 0.47339372,
       0.46990818], shape=(20860,))

### 3. With your DecisionTree module, you must obtain a Gini score of at least 0.1 on the validation dataset.

In [25]:
def my_gini(y_true,y_pred):
    return 2 * roc_auc_score(y_true,y_pred) - 1

### 4. Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?

### Табличка с результатами

In [30]:
dt_sklearn_7 = DecisionTreeClassifier(max_depth=7)
dt_sklearn_12 = DecisionTreeClassifier(max_depth=12)
dt_sklearn_balanced = DecisionTreeClassifier(max_depth=7,class_weight='balanced')
my_dt_7 = MyDecisionTree(max_depth=7, cat_feat = categorical_features)
my_dt_12 = MyDecisionTree(max_depth=12, cat_feat = categorical_features)
my_dt_7_balanced = MyDecisionTree(max_depth=7, cat_feat = categorical_features,criteria='entropy',sample_weights='balanced')

In [31]:
metric_list = [precision_score, recall_score, f1_score,'my_roc_auc']

metric_names = ['precision','recall','f1_score','roc_auc']

name_list = ['dt_sklearn_7', 'dt_sklearn_12', 'dt_sklearn_balanced', 'my_dt_7', 'my_dt_12','my_dt_7_balanced']
model_list = [dt_sklearn_7, dt_sklearn_12, dt_sklearn_balanced, my_dt_7, my_dt_12,my_dt_7_balanced]

In [32]:
def gather_metrics(model_list,name_list,X_train,X_valid,y_train,y_valid):
    
    df_metrics = pd.DataFrame()

    for mod,name in zip(model_list,name_list):
        print(f'model: {name}')

        if 'lgb' in name:
            mod.fit(X_train,y_train,categorical_feature = categorical_features)
        else:
            mod.fit(X_train,y_train)
        
        y_ptr = mod.predict(X_train)
        y_pte = mod.predict(X_valid)
        if 'my' in name:
            y_ptr_prob = mod.predict_proba(X_train)[1].values
            y_pte_prob = mod.predict_proba(X_valid)[1].values

        else:
            y_ptr_prob = mod.predict_proba(X_train)[:,1]
            y_pte_prob = mod.predict_proba(X_valid)[:,1]
        results = {}
        for metric, metric_method in zip(metric_names,metric_list):
            
            if metric == 'roc_auc':
                roc_auc_train= roc_auc_score(y_train,y_ptr_prob)
                roc_auc_test = roc_auc_score(y_valid,y_pte_prob)

                results[metric+'_'+'train'] = roc_auc_train
                results[metric+'_'+'valid'] = roc_auc_test
                results['gini_train'] = 2 * roc_auc_train - 1
                results['gini_valid'] = 2 * roc_auc_test - 1

            else:
                results[metric+'_'+'train'] = metric_method(y_train,y_ptr)
                results[metric+'_'+'valid'] = metric_method(y_valid,y_pte)

        df_metrics.loc[name, results.keys()] = list(results.values())
        print(f'model: {name} Done!')
    return df_metrics


In [33]:
df_metrics = gather_metrics(model_list,name_list,X_train_trans,X_valid_trans,y_train,y_valid)
df_metrics

model: dt_sklearn_7
model: dt_sklearn_7 Done!
model: dt_sklearn_12
model: dt_sklearn_12 Done!
model: dt_sklearn_balanced
model: dt_sklearn_balanced Done!
model: my_dt_7
model: my_dt_7 Done!
model: my_dt_12
model: my_dt_12 Done!
model: my_dt_7_balanced
model: my_dt_7_balanced Done!


,precision_train,precision_valid,recall_train,recall_valid,f1_score_train,f1_score_valid,roc_auc_train,roc_auc_valid,gini_train,gini_valid
dt_sklearn_7,0.864088,0.732051,0.256562,0.206734,0.395649,0.322417,0.749921,0.662984,0.499842,0.325968
dt_sklearn_12,0.957812,0.420762,0.402231,0.248009,0.566543,0.312073,0.84413,0.616959,0.688261,0.233917
dt_sklearn_balanced,0.263217,0.359875,0.610892,0.333816,0.367911,0.346356,0.768672,0.654328,0.537345,0.308655
my_dt_7,0.873362,0.672888,0.262467,0.248009,0.403633,0.362434,0.73967,0.695856,0.47934,0.391712
my_dt_12,0.927536,0.641635,0.335958,0.244388,0.493256,0.353959,0.761649,0.694766,0.523298,0.389532
my_dt_7_balanced,0.285216,0.360628,0.510171,0.390659,0.365882,0.375043,0.750029,0.696852,0.500058,0.393704


### 5. Implement the RandomForestClassifier and check its performance. You have to improve the result of a single tree and get at least 0.15 Gini score on the validation dataset. Be able to set a fixed random seed. 

In [34]:
class MyRandomForest:
    def __init__(self, params, num_trees = 10, random_seed = None, frac = 0.8, subspracing_type = 'sqrt'):

        self.params = params # dict of params for MyDecisionTree
        
        self.num_trees = num_trees
        self.frac = frac
        if subspracing_type not in ['sqrt'] and not isinstance(subspracing_type, int):
            print('subspracing_type must be \'sqrt\' or integer')
            sys.exit(1)
        else:
            self.subspracing_type = subspracing_type
            
        self.is_fitted_ = False

        if random_seed is not None:
            np.random.seed(random_seed)


    def fit(self, X, y):
        self.X = X
        self.y = y
        self.features_ = X.columns
        
        if self.subspracing_type == 'sqrt':
            self.subspracing = int(np.sqrt(len(self.features_)))
        else:
            self.subspracing = self.subspracing_type

        self._build_forest()
        self.is_fitted_ = True
        return self
    
    def _build_forest(self):

        self.forest = []

        for i in range(self.num_trees):
            iter_model = MyDecisionTree(**self.params)        # инициализация модели
            feat_set = self._random_feat_set()                # рандомизация  признаков
            subset = self.X.sample(frac = self.frac)[feat_set]
            iter_model.fit(subset,self.y[subset.index])

            self.forest.append((iter_model,feat_set))

    def _random_feat_set(self):

        return self.features_[np.random.choice(len(self.features_), size=self.subspracing, replace=False)]

    def predict(self, X_test):

        if not self.is_fitted_:
            print('Estimator not fitted')
            sys.exit(1)
        y_pred = self.predict_proba(X_test).idxmax(axis=1)
        
        return y_pred

    def predict_proba(self, X_test):

        if not self.is_fitted_:
            print('Estimator not fitted')
            sys.exit(1)

        all_tree_pred = []
        for tree, feat_set in self.forest:
            tree_pred = tree.predict_proba(X_test[feat_set]).to_numpy()
            all_tree_pred.append(tree_pred)

        avg_pred = np.stack(all_tree_pred).mean(axis=0)     # Оси - (глубина дерева, объекты, целевые переменные)
        df_avg_pred = pd.DataFrame(avg_pred, index=X_test.index, columns=tree.classes_)

        return df_avg_pred

In [35]:
params = {
    'max_depth':7,
    'cat_feat':categorical_features,
    'criteria':'entropy',
    'sample_weights':'balanced'
}
params2 = {
    'max_depth':12,
    'cat_feat':categorical_features
}

my_rf_model_7 = MyRandomForest(params,num_trees=10)
my_rf_model_12 = MyRandomForest(params2,num_trees=50)

rf_sklearn_7 = RandomForestClassifier(max_depth=7,n_estimators=50)
rf_sklearn_12 = RandomForestClassifier(max_depth=12,n_estimators=50)

In [36]:
name_list = ['rf_sklearn_7', 'rf_sklearn_12', 'my_rf_model_7','my_rf_model_12']
model_list = [rf_sklearn_7, rf_sklearn_12, my_rf_model_7,my_rf_model_12]

In [37]:
df_metrics_rf = gather_metrics(model_list,name_list,X_train_trans,X_valid_trans,y_train,y_valid)
df_metrics_rf

model: rf_sklearn_7
model: rf_sklearn_7 Done!
model: rf_sklearn_12
model: rf_sklearn_12 Done!
model: my_rf_model_7
model: my_rf_model_7 Done!
model: my_rf_model_12
model: my_rf_model_12 Done!


c:\school21\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\school21\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,precision_train,precision_valid,recall_train,recall_valid,f1_score_train,f1_score_valid,roc_auc_train,roc_auc_valid,gini_train,gini_valid
rf_sklearn_7,0.930781,0.881279,0.207349,0.139754,0.339147,0.24125,0.781485,0.724747,0.562969,0.449494
rf_sklearn_12,0.993174,0.771631,0.286417,0.196959,0.444614,0.313816,0.915113,0.733521,0.830226,0.467043
my_rf_model_7,0.180047,0.222006,0.676181,0.248371,0.284374,0.23445,0.685498,0.611681,0.370996,0.223361
my_rf_model_12,0.0,0.0,0.0,0.0,0.0,0.0,0.82707,0.712199,0.65414,0.424398


### 6. Use your DecisionTree design class for GBDT classifier. This class must have max_depth, number_of_trees and max_features attributes. You must compute the gradient of the binary cross-entropy loss function and implement incremental learning: train the next tree using the results of the previous trees.

In [38]:
class MyGradientBoostingDecisionTree:

    def __init__(self, params, lr = 0.1, n_trees = 20):
        self.n_trees = n_trees
        self.params = params
        self.lr = lr
        self.is_fitted_ = False

    def fit(self, X, y):
        self.X = X
        self.y = y

        zeros = self.y.value_counts()[0]
        ones = self.y.value_counts()[1]
        
        self.class_weights_ = zeros / ones
        self._build_ensamble()
        self.is_fitted_ = True

        return self

    def _build_naive(self):

        zeros = self.y.value_counts()[0]
        ones = self.y.value_counts()[1]
        self.base_pred_ = np.log(ones / zeros)
        return pd.Series(np.log(ones/zeros), index=self.X.index)
    
    def _sigmoid(self,z):
        # обработка переполнения массива
        return np.where(z>=0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))

    def _build_ensamble(self):

        self.trees = []
        preds = self._build_naive()
        
        for i in range(self.n_trees):

            p = self._sigmoid(preds)

            resideal = (self.y - p) * self.class_weights_

            iter_model = MyDecisionTreeRegressor(**self.params)

            iter_model.fit(self.X, resideal)

            self.trees.append(iter_model)

            preds += self.lr * iter_model.predict(self.X)

    def predict_proba(self, X_test):

        if not self.is_fitted_:
            print('Estimator not fitted')
            sys.exit(1)

        raw_predictions = pd.Series(self.base_pred_, index = X_test.index)
        for tree in self.trees:
            raw_predictions += self.lr * tree.predict(X_test)
        
        p1 = self._sigmoid(raw_predictions)
        p2 = 1 - p1
        proba_df = pd.DataFrame(index=X_test.index)
        proba_df[0] = p2
        proba_df[1] = p1

        return proba_df
    
    def predict(self, X_test):
        y_pred = self.predict_proba(X_test).idxmax(axis=1)
        return y_pred

### 7. Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set. 

Review the documentation of the libraries and fine-tune the algorithms for the task. Note key differences between each implementation.

LightGBM:  
1. Leaf-wise подход. В то время как обычные алгоритмы стоят деревья уровень за уровнем, LightGBM выбирает лист с наибольшим потенциалом снижения функции потерь и разбивает именно его. Этот подход позволяет достигать лучшего качества модели при меньшем количестве итераций,. но требует осторожности при работе с небольшими датасетами из за риска переобучения.
2. GOSS ( Gradient-based One-Side Sampling) - Представляет собо интеллектуальный метод выборки обучающих примеров, который основывается на величине градиентов. Алгоритм сохраняет все примеры с большими градиентами (которые хуже предсказываются моделью) и случайно выбирает только часть примеров с малыми градиентами. Этот подход уменьшает объем данных, необходимых для обучения, при сохранении качества модели, что приводит к существенносму ускорению обучения.
3. EFB (Exclusive Feature Bunding) - решает проблему высокой размерности признаково пространства путем объединения взаимоисключающих признаков в единые "пакеты". Это особенно эфеективно для разреженных данных, где множество признаков имеют нулевые значения. Данная оптимизация значительно сокращает кол-во признаков без потери информации, что ускоряет обучение и снижает потребление памяти.


Catboost:
1. Симметричные деревья (на одно уровне узлов используются одинаковые условия разбиения)
2. Встроенная обработка категориальных признаков
3. Расширяемость. В CatBoost легко добавить свою функцию ошибки
4. Автоматическая обработка пропущенных признаков (но не в категориальных переменных)
5. Поддержка текстовых признаков! Может извлекать признаки из текста используя их для обучения модели.

XGBoost:
1. Второй порядок оптимизации: XGBoost использует информацию о вторых производных (гессиан) функции потерь, что обеспечивает более точную аппроксимацию и быструю сходимость.
2. Регуляризация: Встроенная поддержка L1 и L2 регуляризации предотвращает переобучение на уровне алгоритма.
3. Разреженные данные: Sparsity-aware алгоритм эффективно обрабатывает разреженные данные, автоматически изучая оптимальное направление для пропущенных значений.
4. Параллельные вычисления: Алгоритм построения деревьев распараллелен, что значительно ускоряет обучение.

In [39]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

#### GRIDSEARCH

In [40]:
params_lgb = {
    'n_estimators':[20,50,100],
    'learning_rate':[0.01,0.1,0.15],
    'num_leaves':[10,20,30],
}
gs_lgb = GridSearchCV(LGBMClassifier(verbose = -1),param_grid=params_lgb,scoring='f1')
gs_lgb.fit(X_train_trans,y_train,categorical_feature = categorical_features)
lgbm_best_params = gs_lgb.best_params_

In [41]:
# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 50, 500),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'gamma': trial.suggest_float('gamma', 0, 5),
#         'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
#         'reg_lambda': trial.suggest_float('reg_lambda', 0, 5)
#     }
    
#     model = XGBClassifier(**params, random_state=42)
#     score = cross_val_score(model, X_train_trans, y_train, cv=3, scoring='f1').mean()
#     return score

# # Создание и запуск оптимизации
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=100)

# xgb_best_params = study.best_params
# print("Лучшие параметры:", study.best_params)
# print("Лучший результат:", study.best_value)

In [42]:
params_xgb = {
    'n_estimators':[20,50,100],
    'learning_rate':[0.01,0.1,0.15],
    'max_depth':[5,10,20],
    'booster':[None,'dart']
}
gs_xgb = GridSearchCV(XGBClassifier(),param_grid=params_xgb,scoring='f1')
gs_xgb.fit(X_train_trans,y_train)
xgb_best_params = gs_xgb.best_params_

In [43]:
# model = CatBoostClassifier(cat_features=categorical_features,verbose=0)

# grid = {'learning_rate': [0.03, 0.1],
#         'depth': [4, 8, 15],
#         'l2_leaf_reg': [1, 9, 50],
#         'auto_class_weights':[None,'Balanced']}

# grid_search_result = model.grid_search(grid,
#                                        X=X_train_trans,
#                                        y=y_train,
#                                        plot=True)
# cbm_params = grid_search_result.params

In [44]:
сb_best_params = {'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 1,
        'auto_class_weights':'Balanced'}

In [45]:
print(f'lgbm_best_params {lgbm_best_params}')
print(f'xgb_best_params {xgb_best_params}')
print(f'сb_best_params {сb_best_params}')

lgbm_best_params {'learning_rate': 0.15, 'n_estimators': 50, 'num_leaves': 20}
xgb_best_params {'booster': None, 'learning_rate': 0.15, 'max_depth': 5, 'n_estimators': 100}
сb_best_params {'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 1, 'auto_class_weights': 'Balanced'}


In [46]:
lgb_model = LGBMClassifier(**lgbm_best_params)
xgb_model = XGBClassifier(**xgb_best_params)
cb_model = CatBoostClassifier(**сb_best_params,verbose = 0, cat_features = categorical_features)
params = {
    'max_depth':7,
    'cat_features':categorical_features
}
my_gbdt_model = MyGradientBoostingDecisionTree(params,n_trees=10)

In [47]:
boosting_name_list = ['lgb_model', 'xgb_model', 'cb_model','my_gbdt_model']
boosting_model_list = [lgb_model, xgb_model, cb_model, my_gbdt_model]

In [48]:
df_boost_metrics = gather_metrics(boosting_model_list,boosting_name_list,X_train_trans,X_valid_trans,y_train,y_valid)
df_boost_metrics

model: lgb_model
model: lgb_model Done!
model: xgb_model
model: xgb_model Done!
model: cb_model
model: cb_model Done!
model: my_gbdt_model
model: my_gbdt_model Done!


,precision_train,precision_valid,recall_train,recall_valid,f1_score_train,f1_score_valid,roc_auc_train,roc_auc_valid,gini_train,gini_valid
lgb_model,0.89939,0.712487,0.290354,0.245836,0.438988,0.365545,0.91497,0.70078,0.829939,0.40156
xgb_model,0.900559,0.774115,0.264436,0.229544,0.408826,0.354091,0.869721,0.730767,0.739443,0.461533
cb_model,0.341262,0.388619,0.677822,0.405503,0.453966,0.396882,0.850469,0.728245,0.700937,0.45649
my_gbdt_model,0.934755,0.75939,0.244423,0.234251,0.387516,0.358052,0.774201,0.715053,0.548403,0.430106


 Analyze special features of each algorithm (how does "categorical feature" work in Catboost, what is DART mode in XGBoost)? Which GBDT model gives the best result? Can you explain why?

1. Как работают категориальные признаки в кэтбусте?
Категориальные признаки кодируются с помощью target encoding. Вместо каждого категориального признака заполняется среднее значение целевой переменной
2. Что такое DART мод в XGBoost?
Dropouts meet Multiple Additive Regression Trees - Алгоритм использующий метод dropout для улучшения регуляризации модели и снижения риска переобучения. Он случайным образом удаляет часть деревьев на каждой итерации бустинга, что помогает предотвратить зависимость модели от отдельных деревьев и улучшить её обобщающую способность. 
3. Какая модель даёт лучший результат? Кэтбуст
4. Почему? Потому что кэтбуст использует мощные методы регуляризации и усреднения

### 8. Take the best model and estimate its performance on the test dataset: check the Gini values on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitting or not? Explain.

In [49]:
y_pred_train = cb_model.predict_proba(X_train_trans)[:,1]
y_pred_valid = cb_model.predict_proba(X_valid_trans)[:,1]
y_pred_test = cb_model.predict_proba(X_test_trans)[:,1]

print(f'gini train: {my_gini(y_train,y_pred_train)}')
print(f'gini valid: {my_gini(y_valid,y_pred_valid)}')
print(f'gini test: {my_gini(y_test,y_pred_test)}')

gini train: 0.7009374835993278
gini valid: 0.45649014549397116
gini test: 0.44833730428265217


### 9*. Implement the ExtraTreesClassifier and check its performance. You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.

In [52]:
params = {
    'max_depth':7,
    'cat_feat':categorical_features,
    'split':'random',
    'criteria':'entropy',
    'sample_weights':'balanced'
}

my_extra_trees_clf = MyRandomForest(params,num_trees=50,frac = 1,subspracing_type=len(X_train_trans.columns))

In [53]:
my_extra_trees_clf.fit(X_train_trans,y_train)

y_pred_train = my_extra_trees_clf.predict_proba(X_train_trans)[1]
y_pred_valid = my_extra_trees_clf.predict_proba(X_valid_trans)[1]
y_pred_test = my_extra_trees_clf.predict_proba(X_test_trans)[1]

print(f'gini train: {my_gini(y_train,y_pred_train)}')
print(f'gini valid: {my_gini(y_valid,y_pred_valid)}')
print(f'gini test: {my_gini(y_test,y_pred_test)}')

gini train: 0.4759320096544588
gini valid: 0.4213220539009237
gini test: 0.3217534861135194
